# Lecture 5: Monte Carlo Simulations

---

## 5.0 The method invented at a card table

Stanisław Ulam, a nuclear physicist on the Manhattan Project, was recovering from surgery and playing a lot of solitaire. He wondered: *what is the probability that a randomly dealt game can be won?* Computing it exactly — pure combinatorics — was hopeless. Then the idea: if you cannot *calculate* the probability, **play many games and count**. Doing it by hand would take years, so Ulam asked von Neumann to program it on ENIAC. The method was code-named after the casino where Ulam's uncle gambled: **Monte Carlo**.

The general recipe, for any event $E$ whose probability you cannot compute analytically, generate many random replicates $X_1, \dots, X_M$ of the experiment. To estimate the probability $P[E]$, count the experiments in which $E$ occurred and divide by the total number of experiments, $M$:

$$ \frac{1}{M} \sum_{i=1}^M \mathbb{1}_{X_i \in E}$$

By the **Law of Large Numbers**, this fraction converges to the true $P[E]$ as $M \to \infty$. That is the entire method. Its power is that "the experiment" can be anything you can *simulate*: a solitaire deal, a path of stock returns or your own retirement account.




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(17)     # reproducibility: same "random" numbers on every run

---
## 5.1 Idea behind Monte Carlo Simulations: Computing $\pi$ by throwing darts

We know that the area of a circle with radius $r = 1$ is $\pi r^2 = \pi$. Then the area of a quarter of a circle is $\pi/4$. Let us place a quarter circle into a unit square. Next we are going to "shoot" random points at the unit square. Each of these points can be represented by coordinates $0 \leq x \leq 1$ and $0 \leq y \leq 1$.

A point is inside the quarter circle if $x^2 + y^2 \leq 1$. The probability of a point being inside the circle equals the ratio of the areas

$P(in \; circle) = \frac{Area\;circle}{Area\;square} = \frac{\pi/4}{1} = \pi/4$

Then we can compute Monte Carlo approximation of $\pi$ as

$\pi = 4 \times \frac{Number\; of\; points\; inside\; circle}{Total\; Number\; of\; points}$

### Predict

With **1,000 points**, how accurate will the estimate of $\pi = 3.14159...$ be? (a) usually right to ~3 decimals (b) usually right to ~1-2 decimals (c) often wrong already in the first decimal

In [ ]:
def compute_pi(N_points):
    x = np.random.random(size=N_points)
    y = np.random.random(size=N_points)
    N_circle = np.sum(x**2 + y**2 <= 1)      # count points inside the circle
    return 4 * N_circle / N_points

compute_pi(1000)

In [ ]:
# visualize the dartboard
N_points = 10000
x = np.random.random(size=N_points)
y = np.random.random(size=N_points)

in_circle = x**2 + y**2 <= 1
plt.scatter(x[in_circle], y[in_circle], s=10)
plt.scatter(x[~in_circle], y[~in_circle], s=10)
x_c = np.linspace(0, 1, 150)
plt.plot(x_c, np.sqrt(1 - x_c**2), 'k')
plt.xlim([0, 1]); plt.ylim([0, 1]);

### The Law of Large Numbers

Run the estimate for growing $M$ and plot the error.

In [ ]:
Ms = [100, 1_000, 10_000, 100_000, 1_000_000]
for M in Ms:
    estimates = np.array([compute_pi(M) for _ in range(20)])
    print(f"M = {M:>9,}:  mean estimate {estimates.mean():.4f},  "
          f"typical error {estimates.std():.4f}")

<details>
<summary> <b>Explanation</b></summary>

Each ×100 in draws cuts the error by only ×10: Monte Carlo error shrinks like $1/\sqrt{M}$. That answers the Moodle question (10× precision needs **100×** the draws) and sets the working rule for your assignment: 10,000+ paths is cheap, so use them — but no number of paths fixes a wrong *model*. Precision ≠ accuracy: a simulation converges beautifully to the answer implied by its assumptions, right or wrong.

</details>

**Practice:**

Compute $\pi$ 10000 times (using 1000 points for each simulation) and report its average. Is it closer to the true value?

In [ ]:
# solution


<details>
<summary> <b>Solution</b></summary>

```python
N_sim = 10000
N_points = 1000

pi_est = np.array([compute_pi(N_points) for _ in range(N_sim)])
np.mean(pi_est)
```
</details>



---
## 5.2 Choosing the randomness: distributions

A simulation is only as good as the distribution driving it. The standard toolbox:

**Uniform** — all values in a range equally likely; the "I only know the range" distribution. *Example: an analyst covering a new fintech has seen cash flows of 100 and 200 mln and has no basis for anything finer:*

In [ ]:
CFs = np.random.uniform(100, 200, size=500)
plt.hist(CFs); plt.title("Simulated cash flows, uniform(100, 200)");

**Bernoulli / Binomial** — a (possibly repeated) coin flip with success probability $p$. Perfect for **states**: recession or boom. We estimate $p$ from a century and a half of US recession data (FRED series `USREC`):

<details>
<summary> <b>Formal definition</b></summary>

The random variable $X$ with the **Bernoulli distribution** defined by the parameter $p$ represents an outcome of a trial or a coin toss, where we observe success with probability $p$.

$X =  1$,  with probability $p$,  $X = 0$, with probability $1-p$

The random variable $X$ with the Binomial distribution defined by the parameters $p$ and $N$ represents an outcome of a series of $N$ trials or a coin tosses, where we observe success in each trial with probability $p$.

$Y = \sum_{i=1}^{N} X_i,$ where each $X_i$ has the Bernoulli distribution.
</details>

In [ ]:
# NBER recession indicator via FRED (annual, 1855-2025)
url = ('https://fred.stlouisfed.org/graph/fredgraph.csv?id=USREC'
       '&cosd=1855-01-01&coed=2025-01-01&fq=Annual&fam=eop&nd=1854-12-01')
data = pd.read_csv(url, index_col=0, parse_dates=True)

p_recession = np.sum(data['USREC']) / data.shape[0]
print(f"Unconditional P(recession) = {p_recession:.3f}")

# simulate 100 years of independent boom/recession draws
x = np.random.binomial(n=1, p=p_recession, size=100)
print(x)
print("Fraction of recession years in the simulation:", x.mean())

**Normal** (`np.random.normal(loc, scale)`) — the default model for stock returns; **Student-t** (`np.random.standard_t(df)`) — for *fat-tailed* returns; **Poisson** — arrivals of rare events.

### Discussion — does independence look right?

Stare at the simulated 0/1 sequence above and at reality: US recessions arrive in *clumps* of consecutive years. What is the independent-coin model missing?

### Markov chains

The fix: let this year's state depend on last year's. A **Markov chain** is defined by a transition matrix — $P_{ij}$ = probability of moving to state $j$ given you are in state $i$. Estimate the transition probabilities from the data:

In [ ]:
# conditional probabilities from the historical sequence
p_r_r = np.sum((data['USREC'] == 1) & (data['USREC'].shift(-1) == 1)) / np.sum(data['USREC'] == 1)
p_b_b = np.sum((data['USREC'] == 0) & (data['USREC'].shift(-1) == 0)) / np.sum(data['USREC'] == 0)
p_r_b = 1 - p_r_r
p_b_r = 1 - p_b_b
print(f"P(recession next year | recession now) = {p_r_r:.2f}")
print(f"P(boom next year      | boom now)      = {p_b_b:.2f}")

Now that we created a transition matrix, let us simulate the state of the economy for $T = 100$ periods.

In [ ]:
def model_economy_states(transition_matrix, T):
  N = transition_matrix.shape[0]
  states = range(N)
  T_start = 100                      # burn-in so the start state doesn't matter
  economy_state = np.zeros(T + T_start).astype(int)
  for t in range(1, T + T_start):
    economy_state[t] = np.random.choice(states, size=1, p=transition_matrix[economy_state[t-1], :])[0]
  return economy_state[T_start:]

transition_matrix = np.array([[p_b_b, p_b_r],
                              [p_r_b, p_r_r]])
economy_states = model_economy_states(transition_matrix, 100)
plt.plot(economy_states); plt.yticks([0, 1], ['boom', 'recession']);

Now the recessions clump, as in reality. **Modelling choice made visible:** iid coin vs. Markov chain is exactly the kind of "simulation design" decision your assignment's Monte Carlo section must state and justify.

---
## 5.3 Simulating retirement

Lecture 1's retirement problem had one simplification: it assumed the account earns **exactly 7% every single year**. Markets do not work like that. Historically, a broad equity portfolio has returned *around* 7% per year on average — with a standard deviation of roughly 18%. How much can the volatility matter over a working life?

**Retirement plan:** invest **€6,000 per year** (€500/month) for **40 years** with returns drawn from the normal distribution $N(7\%, 18\%)$.

### Prediction

If the interest rate stayed at exactly 7%, the final wealth would be about **€1.28 million**. In the simulation:

1. The **average** simulated final wealth will be: (a) ≈ €1.28M (b) clearly above (c) clearly below
2. The **median** — the outcome of the middle saver: (a) ≈ the mean (b) noticeably below the mean (c) noticeably above
3. The probability of ending **below** the deterministic €1.28M: (a) ~50% (b) ~65% (c) ~35%

In [ ]:
np.random.seed(2270)
n_sim = 10_000
T = 40
contribution = 6_000
mu = 0.07
sigma = 0.18

final_wealth = np.zeros(n_sim)
paths = np.zeros((n_sim, T))
for i in range(n_sim):
    wealth = 0.0
    r = np.random.normal(mu, sigma, T)      # 40 annual returns
    for t in range(T):
        wealth = (wealth + contribution) * (1 + r[t])
        paths[i, t] = wealth
    final_wealth[i] = wealth

# the deterministic benchmark from Lecture 1
wealth_det = 0.0
for t in range(T):
    wealth_det = (wealth_det + contribution) * (1 + mu)

print(f"Deterministic (7% every year):  {wealth_det:,.0f} EUR")
print(f"Mean of simulations:            {final_wealth.mean():,.0f} EUR")
print(f"Median of simulations:          {np.median(final_wealth):,.0f} EUR")
print(f"5th percentile:                 {np.percentile(final_wealth, 5):,.0f} EUR")
print(f"95th percentile:                {np.percentile(final_wealth, 95):,.0f} EUR")
print(f"P(final wealth < deterministic): {np.mean(final_wealth < wealth_det):.1%}")

Figure to illustrate the distribution of possible outcomes

In [ ]:
# the fan of possible lives: percentile bands of the wealth path over time
years = np.arange(1, T + 1)
# area between 5th and 95th percentile
plt.fill_between(years, np.percentile(paths, 5, axis=0),
                     np.percentile(paths, 95, axis=0), alpha=0.15, color='C0')
# area between 25th and 75th percentiles
plt.fill_between(years, np.percentile(paths, 25, axis=0),
                     np.percentile(paths, 75, axis=0), alpha=0.3, color='C0')
# median
plt.plot(years, np.percentile(paths, 50, axis=0), 'C0', label='median')
# deterministic path
det = np.zeros(T); w = 0.0
for t in range(T):
    w = (w + contribution) * (1 + mu); det[t] = w
plt.plot(years, det, 'k--', label='deterministic 7%')
plt.legend(); plt.xlabel('year'); plt.ylabel('EUR')
plt.title('40 years of saving 6,000 EUR/yr: the fan of outcomes');

<details>
<summary> <b>Explanation</b></summary>

The **mean** of the simulations lands near the deterministic value — averages behave. But the **median** sits far below it, and **roughly two-thirds of simulated lives end below the "expected" €1.28M**. Compounding is multiplicative, so the distribution of final wealth is right-skewed: a minority of spectacular paths drags the mean up while the typical path lags it. The number a pension brochure shows you ("assuming 7% annual returns...") is not the outcome the typical saver gets.

And look at the 5th percentile: same saver, same discipline, same *average* market — several times less money. That spread is **the risk that deterministic planning hides**, and showing it to a client honestly is precisely what Monte Carlo simulations are for.

</details>

### Homework

**Sequence-of-returns risk:** take one simulated 40-year return sequence; compute final wealth with the returns in the given order, then with the *same returns sorted worst-first, then best-first* (`np.sort()`). Same average return, same everything — how different is final wealth, and why does order matter here when it never mattered for a lump sum? *(Hint: with contributions, late returns act on much more money than early ones.)*

<details>
<summary> <b>Solution sketch</b></summary>

```python
r = np.random.normal(mu, sigma, T)
def wealth_path(r):
    w = 0.0
    for x in r:
        w = (w + contribution) * (1 + x)
    return w
print(f"as drawn:    {wealth_path(r):,.0f}")
print(f"worst first: {wealth_path(np.sort(r)):,.0f}")
print(f"best first:  {wealth_path(np.sort(r)[::-1]):,.0f}")
```

Worst-first is the *good* ordering for a saver (crashes hit while the pot is small; booms hit the big pot) — for a **retiree withdrawing**, exactly the reverse: early crashes are lethal. One distribution, opposite risks depending on the cash-flow direction — a genuinely deep fact your clients' age structure determines.

</details>


---
## 5.4 Application: a simple leveraged buyout (LBO)


In a **leveraged buyout**, a private equity fund buys a company using mostly **borrowed money**. The recipe: buy at some multiple of EBITDA; let the company's cash flow **pay down the debt** during the holding period; sell ("exit") after a few years at — one hopes — a higher multiple of a higher EBITDA. Because the fund's equity is a thin slice of the purchase price, every euro of value created (or destroyed) lands on it, amplified.

**The deal — TascaCo**, a (fictional) restaurant-supply company:

| Item | Value |
|---|---|
| EBITDA today | €100M, growing with the economy |
| Purchase price | 9 × EBITDA = **€900M** enterprise value |
| Financing | **€540M debt**  at 8% interest + **€360M equity** |
| Cash conversion | 60% of EBITDA becomes free cash flow before interest; all of it sweeps debt |
| Exit | year 5, at **10 × EBITDA** (the base case assumes *multiple expansion*) |
| Fund hurdle | 15% IRR to equity |

Here, cash conversion is a catch-all parameter. In practice, it accounts for cash taxes (roughly 20%-25% of EBIT), maintenance and growth capex (often 10%-20% of EBITDA for a mid-market industrial or distribution business like our fictional TascaCo), and investment in net working capital (a growing company ties up cash in receivables and inventory).


The investment committee's deck models this the classic way — one deterministic **base case**, EBITDA growing 4% every year:


In [ ]:
# --- the deck's base case: one path, no uncertainty -------------------
EBITDA = 100             # EBITDA (EUR millions)
e_growth = 0.04          # EBITDA grows 4% -- every year, apparently
debt =  540.0            # debt (EUR millions)
equity_in = 360.0
cash_conversion = 0.6    # EBITDA less cash taxes, capex, and working-capital needs
interest_rate = 0.08
exit_multiple = 10
n_years = 5

for t in range(n_years):
    EBITDA = EBITDA * (1+e_growth)
    fcf = cash_conversion*EBITDA - interest_rate*debt   # free cash flow after interest
    debt = max(debt - fcf, 0)               # cash sweep: repay debt
    print(f"year {t+1}: EBITDA {EBITDA:6.1f} debt {debt:6.1f}")

equity_exit = exit_multiple*EBITDA - debt        # exit at 10x, equity gets what's left after debt
irr_base = (equity_exit / equity_in)**(1/n_years) - 1
print(f"\nExit equity value: {equity_exit:.0f}  ->  base-case IRR: {irr_base:.1%}")


**17.8%** — comfortably above the 15% hurdle. The deck recommends committing the €360M.

That is **one path**. EBITDA will **not** grow 4.0% five years running, and nobody promises a $10\times$ exit. So we replace the two hopeful constants with distributions and let the Markov economy from Section 5.2 drive growth: in booms EBITDA grows $N(+8.2\%, 8\%)$, in recessions $N(-6\%, 8\%)$; the exit multiple is $N(10\times, 1.5\times)$.

### Predict

1. The **median** simulated IRR, relative to the deck's 17.8%: (a) about equal (b) noticeably below (c) noticeably above
2. The probability the equity **loses money** (IRR < 0): (a) ~3% (b) ~12% (c) ~25%
3. Can the equity be **wiped out completely** — exit value below the remaining debt? (a) essentially never (b) rarely but yes (c) ~1 deal in 10


In [ ]:
# --- the same deal, 10,000 possible worlds ----------------------------
np.random.seed(2270)
n_sim = 10_000
hurdle = 0.15
cash_conversion = 0.6
interest_rate = 0.08
exit_mult_avg = 10
exit_mult_std = 1.5
exit_mult_fire_sale = 5       # floor at a 5x fire sale
n_years = 5
EBITDA = 100
e_growth_avg = np.array([0.082,-0.06])   # 0 - boom, 1 - recession
e_growth_std = 0.08
debt =  540.0
equity_in = 360.0

irr = np.zeros(n_sim)
wipeouts = 0
for i in range(n_sim):
    # model states of the economy
    states = model_economy_states(transition_matrix, n_years)
    # starting values at time 0
    ebitda = EBITDA
    d = debt
    for t in range(n_years):
        e_growth = np.random.normal(e_growth_avg[states[t]], e_growth_std)
        ebitda = ebitda * (1 + e_growth)
        fcf = cash_conversion*ebitda - interest_rate*d    # if negative, debt grows (a revolver draw)
        d = max(d - fcf, 0)
    exit_multiple = max(np.random.normal(exit_mult_avg, exit_mult_std), exit_mult_fire_sale)
    equity_exit = max(exit_multiple*ebitda - d, 0)             # equity cannot go below 0...
    if equity_exit > 0:
        irr[i] = (equity_exit / equity_in)**(1/n_years) - 1
    else:
        wipeouts += 1
        irr[i] = -1.0                                     # ...it just goes to 0: -100%

print(f"Deck's base-case IRR:        {irr_base:.1%}")
print(f"Median simulated IRR:        {np.median(irr):.1%}")
print(f"P(IRR < 0)  -- lose money:   {np.mean(irr < 0):.1%}")
print(f"P(total equity wipeout):     {wipeouts/n_sim:.2%}")
print(f"P(IRR > {hurdle:.0%} hurdle):        {np.mean(irr > hurdle):.1%}")

plt.hist(irr[irr > -1], bins=50)
plt.axvline(irr_base, color='k', linestyle='--', label='deck base case')
plt.axvline(hurdle, color='r', linestyle=':', label='hurdle')
plt.xlabel('equity IRR'); plt.legend(); plt.title('One deal, 10,000 worlds');


<details>
<summary><b>Explanation</b></summary>

Three things the deck could not show. First, the median IRR lands almost exactly on the base case: the deck's central estimate was fine. That is the point: the base case was never the problem, it's everything the base case leaves out.
Second, clearing the hurdle, which the deck presented as the outcome, happens in only about six worlds out of ten.
Third, the left tail is sizeable: about one deal in ten loses money, and in a small slice of worlds the exit value does not even cover the remaining debt, so the equity is a total loss. That outcome literally does not exist in the deterministic model.

</details>

### So is the debt the problem?

Same company, same worlds — but bought **with no debt at all**: €900M of equity, all free cash flow accumulating instead of repaying anyone. Before running: which version has the higher *median* IRR? Which has the smaller *probability of losing money*?


In [ ]:
# --- the unlevered counterfactual: same company, no debt --------------
np.random.seed(2270)                      # same worlds as above
n_years = 5
EBITDA = 100
e_growth_avg = np.array([0.082,-0.06])   # 0 - boom, 1 - recession
e_growth_std = 0.08
exit_mult_avg = 10
exit_mult_std = 1.5
exit_mult_fire_sale = 5       # floor at a 5x fire sale
equity_in = 900
irr_unlev = np.zeros(n_sim)
for i in range(n_sim):
    states = model_economy_states(transition_matrix, n_years)
    ebitda = EBITDA
    cash = 0.0
    for t in range(n_years):
        e_growth = np.random.normal(e_growth_avg[states[t]], e_growth_std)
        ebitda = ebitda * (1 + e_growth)
        cash = cash + cash_conversion*ebitda               # FCF piles up instead of repaying debt
    exit_multiple = max(np.random.normal(exit_mult_avg, exit_mult_std), exit_mult_fire_sale)
    irr_unlev[i] = ((exit_multiple*ebitda + cash) / equity_in)**(1/n_years) - 1

print("                    levered    unlevered")
print(f"median IRR:          {np.median(irr):6.1%}     {np.median(irr_unlev):6.1%}")
print(f"P(IRR < 0):          {np.mean(irr < 0):6.1%}     {np.mean(irr_unlev < 0):6.1%}")
print(f"5th percentile IRR:  {np.percentile(irr, 5):6.1%}     {np.percentile(irr_unlev, 5):6.1%}")


### Takeaway

The company is identical in both runs; only the financing differs. Leverage **added several points of median IRR** — and multiplied the probability of losing money severalfold, turning a mild 5th percentile into a deep one. **Leverage does not create value; it stretches the distribution** — a richer middle and right, a fatter and longer left tail — and the LBO fee model is largely a claim on the right side of that stretch, while the left side belongs to whoever holds the equity when the music stops.

### Explore

1. **Crank the leverage** to 75% (debt €675M, equity €225M). Median IRR and P(wipeout) both move — by how much? Roughly where does P(IRR < 0) cross 25%?
2. **Correlate the exit with the economy:** if year 5 ends in recession, subtract 2× from the exit multiple (you exit when you must, not when you want). What happens to the left tail, and why is assuming an *independent* exit multiple quietly optimistic?

<details>
<summary> <b>Solution sketch</b></summary>

(1) Change `540.0` to `675.0` and `360.0` to `225.0` and rerun. Median IRR rises a few points; the wipeout probability rises roughly tenfold — the same stretch, stretched further.

(2) Change the definition `exit_mult_avg = np.array([10,8])` and `exit_multiple = max(np.random.normal(exit_mult_avg[states[-1]], exit_mult_std), exit_mult_fire_sale)`. The left tail thickens noticeably: bad operations and bad exit prices now arrive *together*.

</details>





---
## 5.5 The AI wrote it — should you trust it?

A student asked an AI to *"simulate 10 years of monthly returns for a stock with 7% expected annual return and 18% annual volatility, and estimate the probability of losing money over 10 years."* It answered:

In [ ]:
# --- AI-generated code: DO NOT TRUST -- find the bug ------------------
n_sim, n_months = 10_000, 120
mu_annual, sigma_annual = 0.07, 0.18

monthly_rets = np.random.normal(mu_annual/12, sigma_annual/12, (n_sim, n_months))
final = (1 + monthly_rets).prod(axis=1)
print(f"P(lose money over 10 years) = {np.mean(final < 1):.2%}")

<details>
<summary> <b>Solution</b></summary>

`sigma_annual/12` is wrong: means scale with time, but **volatility scales with the square root of time**. Monthly volatility is $0.18/\sqrt{12} \approx 5.2\%$, not $1.5\%$. The AI's simulated world is three and a half times too calm, so the printed probability of loss is far too low — dangerously so, if a client decision depends on it.

The plausibility check that catches it without reading a line: a near-zero chance of losing money in equities over a decade contradicts everything you know (the 2000s alone: two halvings). **If a simulation says risk is (almost) zero, the model is broken until proven otherwise.**

```python
monthly_rets = np.random.normal(mu_annual/12, sigma_annual/np.sqrt(12), (n_sim, n_months))
final = (1 + monthly_rets).prod(axis=1)
print(f"P(lose money over 10 years) = {np.mean(final < 1):.2%}")
```

With the correct scaling, the probability of losing money over 10 years is about 17%, roughly one in six. The bug turned a one-in-six risk into 'impossible.'
</details>

---
